<a href="https://colab.research.google.com/github/Dharshini1701/priyadharshini/blob/main/sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas openpyxl

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving orders_staging.xlsx.zip to orders_staging.xlsx.zip


In [ ]:
import os

print(os.listdir())

['.config', 'orders_staging.xlsx.zip', 'orders_staging', 'sample_data']


In [ ]:
import zipfile

with zipfile.ZipFile("orders_staging.xlsx.zip", "r") as zip_ref:
    zip_ref.extractall("orders_staging")

print(os.listdir("orders_staging"))

['orders_raw.xlsx', 'SQLite_Data_Cleaning_Classroom_Guide.docx']


In [ ]:
import pandas as pd

df = pd.read_excel("orders_staging/orders_raw.xlsx")

print(df.head())

   OrderID     CustomerName                         Email           Phone  \
0     3105       Jaya Verma        jaya.verma801@mail.com    733-307-7375   
1     6354     Farhan Yadav  farhan.yadav65@company.co.in     23114 80745   
2     8690    Chitra Reddy   chitra.reddy695company.co.in    379-490-1038   
3     5858       Sneha Iyer   sneha.iyer945@company.co.in  +91 8472931687   
4     6012    kabir kumar      kabir.kumar305@outlook.com      0400977061   

            City   OrderDate   Amount     Status  
0          delhi  2024-01-05  3091.09    Pending  
1      Bengaluru  04-14-2023  1662.89   Refunded  
2           pune  2024-08-03  1610.32   Refunded  
3   Coimbatore    11-08-2023  2265.32    Pending  
4    Coimbatore   11-19-2024  2718.23  Cancelled  


In [ ]:
import sqlite3
conn=sqlite3.connect("orders.db")
cursor=conn.cursor()

In [ ]:
df.to_sql("orders_raw", conn, if_exists="replace", index=False)

10000

In [ ]:
cursor.execute("""
SELECT name FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())

[('orders_raw',)]


In [ ]:
cursor.execute("delete from orders_raw where orderID not in (select min(orderID)from orders_raw group by OrderID,Email,Phone,City,OrderDate,Amount,Status)")
for row in cursor.fetchall():
  print(row)

In [ ]:
cursor.execute("""update orders_raw set
CustomerName=Trim(CustomerName),
Email=Trim(Email),
Phone=Trim(Phone),
City=Trim(City),
OrderDate=Trim(OrderDate),
Status=Trim(Status),
Amount=trim(Amount) """),conn


(<sqlite3.Cursor at 0x7a52f71e7340>, <sqlite3.Connection at 0x7a52f713f6a0>)

In [ ]:
cursor.execute("""
SELECT CustomerName, Email, Phone, City, OrderDate, Amount, Status
FROM orders_raw
LIMIT 5
""")
for row in cursor.fetchall():
  print(row)

('Jaya Verma', 'jaya.verma801@mail.com', '733-307-7375', 'delhi', '2024-01-05', '3091.09', 'Pending')
('Farhan Yadav', 'farhan.yadav65@company.co.in', '23114 80745', 'Bengaluru', '04-14-2023', '1662.89', 'Refunded')
('Chitra Reddy', 'chitra.reddy695company.co.in', '379-490-1038', 'pune', '2024-08-03', '1610.32', 'Refunded')
('Sneha Iyer', 'sneha.iyer945@company.co.in', '+91 8472931687', 'Coimbatore', '11-08-2023', '2265.32', 'Pending')
('kabir kumar', 'kabir.kumar305@outlook.com', '0400977061', 'Coimbatore', '11-19-2024', '2718.23', 'Cancelled')


In [ ]:
cursor.execute("""SELECT CustomerName
FROM orders_raw
LIMIT 10""")
for row in cursor.fetchall():
  print(row)

('Jaya Verma',)
('Farhan Yadav',)
('Chitra Reddy',)
('Sneha Iyer',)
('kabir kumar',)
('Kabir Joshi',)
('Deepa Chatterjee',)
('Esha Iyer',)
('Arun Yadav',)
('qadir singh',)


In [ ]:
df = pd.read_excel("orders_staging/orders_raw.xlsx")
df.to_sql("orders_raw", conn, if_exists="replace", index=False)

10000

In [ ]:
cursor.execute("SELECT CustomerName FROM orders_raw LIMIT 10")
print(cursor.fetchall())

[('Jaya Verma',), ('Farhan Yadav',), ('  Chitra Reddy ',), (' Sneha Iyer',), ('  kabir kumar  ',), ('Kabir Joshi ',), ('Deepa Chatterjee  ',), ('Esha Iyer',), ('Arun Yadav ',), (' qadir singh  ',)]


In [ ]:
cursor.execute("""
SELECT CustomerName,
       INSTR(CustomerName, ' ') AS space_position
FROM orders_raw
LIMIT 10
""")

print(cursor.fetchall())

[('Jaya Verma', 5), ('Farhan Yadav', 7), ('  Chitra Reddy ', 1), (' Sneha Iyer', 1), ('  kabir kumar  ', 1), ('Kabir Joshi ', 6), ('Deepa Chatterjee  ', 6), ('Esha Iyer', 5), ('Arun Yadav ', 5), (' qadir singh  ', 1)]


In [ ]:
cursor.execute("""
UPDATE orders_raw
SET CustomerName = TRIM(CustomerName)
WHERE CustomerName IS NOT NULL
""")

conn.commit()

In [ ]:
cursor.execute("""
SELECT CustomerName,
       INSTR(CustomerName, ' ') AS space_position
FROM orders_raw
LIMIT 10
""")

print(cursor.fetchall())

[('Jaya Verma', 5), ('Farhan Yadav', 7), ('Chitra Reddy', 7), ('Sneha Iyer', 6), ('kabir kumar', 6), ('Kabir Joshi', 6), ('Deepa Chatterjee', 6), ('Esha Iyer', 5), ('Arun Yadav', 5), ('qadir singh', 6)]


In [ ]:
cursor.execute("""
UPDATE orders_raw
SET CustomerName =
    UPPER(SUBSTR(CustomerName,1,1)) ||
    LOWER(SUBSTR(CustomerName,2,INSTR(CustomerName,' ')-2)) ||
    ' ' ||
    UPPER(SUBSTR(CustomerName,INSTR(CustomerName,' ')+1,1)) ||
    LOWER(SUBSTR(CustomerName,INSTR(CustomerName,' ')+2))
WHERE CustomerName IS NOT NULL
  AND INSTR(CustomerName,' ') > 0
""")

conn.commit()

In [ ]:
cursor.execute("""SELECT CustomerName
FROM orders_raw
LIMIT 10""")
for row in cursor.fetchall():
  print(row)

('Jaya Verma',)
('Farhan Yadav',)
('Chitra Reddy',)
('Sneha Iyer',)
('Kabir Kumar',)
('Kabir Joshi',)
('Deepa Chatterjee',)
('Esha Iyer',)
('Arun Yadav',)
('Qadir Singh',)


In [ ]:
cursor.execute("""update orders_raw
set City=case upper(Trim(City))
when 'CHENNAI' THEN 'Chennai'
when 'BANGALORE' THEN 'Bangalore'
when 'COIMBATORE' THEN 'Coimbatore'
when 'MUMBAI' THEN 'Mumbai'
when 'PUNE' THEN 'Pune'
when 'KOLKATA' THEN 'Kolkata'
Else City
end
where City is not null""")
conn.commit()


In [ ]:
cursor.execute("select City from orders_raw limit 5")
for row in cursor.fetchall():
  print(row)

(' delhi',)
('Bengaluru',)
('Pune',)
('Coimbatore',)
('Coimbatore',)


In [ ]:
cursor.execute("""
update orders_raw set Email=lower(Email) where Email is not null""")
conn.commit()

In [ ]:
cursor.execute("select Email  from orders_raw limit 5")
for row in cursor.fetchall():
  print(row)

('jaya.verma801@mail.com',)
('farhan.yadav65@company.co.in',)
('chitra.reddy695company.co.in',)
('sneha.iyer945@company.co.in',)
('kabir.kumar305@outlook.com',)


In [ ]:
cursor.execute("""update orders_raw
set Status=case upper(Trim(status))
when 'COMPLETED' Then 'Completed'
when 'CANCELLED' Then 'Cancelled'
when 'PENDING' Then 'pending'
when 'REFUNDED' Then 'Refunded'
when 'N/A' Then Null
when ''    Then Null
Else Status
end""")
conn.commit()


In [ ]:
cursor.execute("select Status from orders_raw limit 5")
for row in cursor.fetchall():
  print(row)

('pending',)
('Refunded',)
('Refunded',)
('pending',)
('Cancelled',)


In [ ]:
cursor.execute("""update orders_raw set CustomerName= 'Unknown Customer' where CustomerName is Null""")
cursor.execute("update orders_raw set City = 'Unknown' where City is Null")
cursor.execute("update orders_raw set Status = 'Unknown' where status is Null")
conn.commit()

In [ ]:
cursor.execute("""SELECT CustomerName, City, Status
FROM orders_raw
LIMIT 10
""")
for row in cursor.fetchall():
  print(row)


('Jaya Verma', ' delhi', 'pending')
('Farhan Yadav', 'Bengaluru', 'Refunded')
('Chitra Reddy', 'Pune', 'Refunded')
('Sneha Iyer', 'Coimbatore', 'pending')
('Kabir Kumar', 'Coimbatore', 'Cancelled')
('Kabir Joshi', '  Delhi', 'pending')
('Deepa Chatterjee', 'Chennai', 'Refunded')
('Esha Iyer', ' HYDERABAD ', 'pending')
('Arun Yadav', 'Mumbai', 'Completed')
('Qadir Singh', 'Delhi ', 'pending')


In [ ]:
df.to_sql("orders_raw", conn, if_exists="replace", index=False)

NameError: name 'df' is not defined

In [ ]:
cursor.execute("""
UPDATE orders_raw
SET OrderDate = CASE
    WHEN OrderDate LIKE '____-__-__' THEN
        OrderDate

    WHEN OrderDate LIKE '__-__-____' THEN
        SUBSTR(OrderDate,7,4) || '-' ||
        SUBSTR(OrderDate,1,2) || '-' ||
        SUBSTR(OrderDate,4,2)

    WHEN OrderDate LIKE '__/__/____' THEN
        SUBSTR(OrderDate,7,4) || '-' ||
        SUBSTR(OrderDate,4,2) || '-' ||
        SUBSTR(OrderDate,1,2)

    ELSE NULL
END
WHERE OrderDate IS NOT NULL
""")

conn.commit()

In [ ]:
cursor.execute("""
SELECT OrderDate
FROM orders_raw
LIMIT 10
""")

for row in cursor.fetchall():
  print(row)

('2024-01-05',)
('2023-04-14',)
('2024-08-03',)
('2023-11-08',)
('2024-11-19',)
('2023-02-05',)
('2024-12-02',)
('2025-02-28',)
('2025-02-05',)
('2025-07-24',)


In [ ]:
cursor.execute("""alter table orders_raw add column Email_valid integer""")
conn.commit()

In [ ]:
cursor.execute("""update orders_raw set Email_Valid = case
when Email is null Then 0
when Email like '%_@_%.__%' and Email not like '% %' Then 1
else 0
end""")
conn.commit()

In [ ]:
cursor.execute("select Email from orders_raw limit 10")
for row in cursor.fetchall():
  print(row)

('jaya.verma801@mail.com',)
('farhan.yadav65@company.co.in',)
('chitra.reddy695company.co.in',)
('sneha.iyer945@company.co.in',)
('kabir.kumar305@outlook.com',)
('kabir.joshi17@mail.com',)
('deepa.chatterjee695@outlook.com',)
('esha.iyer391@gmail.com',)
('arun.yadav764@mail.com',)
('qadir.singh378@mail.com',)


In [ ]:
cursor.execute("""update orders_raw  set Phone=replace(replace(replace(replace(replace(Phone,'(',''),')',''),'-',''),' ',''),'+','')where Phone is not null
""")
cursor.execute("""update orders_raw set Phone=substr(Phone,3)where Phone is not null and length(Phone)=12 and substr(Phone,1,2)='91'
""")
cursor.execute("""alter table orders_raw add column Phone_valid integer""")
cursor.execute("""update orders_raw set Phone_valid = case when Phone is not null and length(Phone)=10 and Phone glob '[0-9]*' Then 1
else 0
End
""")
conn.commit()


In [ ]:
cursor.execute("""select phone from orders_raw limit 10""")
for row in cursor.fetchall():
  print(row)

('7333077375',)
('2311480745',)
('3794901038',)
('8472931687',)
('0400977061',)
('7565619112',)
('1238943682',)
('7692743811',)
('4798431264',)
('4311621065',)


In [ ]:
cursor.execute("""update orders_raw set Amount = replace(replace(replace(Amount,'₹',''),',',''),'','') where Amount is not Null """)
cursor.execute("""update orders_raw set Amount=null where Amount='N/A' or Amount=''""")
cursor.execute("""update orders_raw  set Amount_Clean=cast(Amount as real)where Amount is not null""")
cursor.execute(""" alter table Orders_raw add column Amount_Flag text""")
cursor.execute("""update orders_raw set Amount_Flag=case when Amount_Clean is null then 'Missing'
when Amount_clean <0 Then 'negative'
when Amount_clean >20000 Then 'outlier'
else 'ok'
end""")
conn.commit()

In [ ]:
cursor.execute("select Amount_Clean from orders_raw limit 10")
for row in cursor.fetchall():
  print(row)

(3091.09,)
(1662.89,)
(1610.32,)
(2265.32,)
(2718.23,)
(434.17,)
(2333.7,)
(3818.14,)
(1297.01,)
(2274.49,)


In [ ]:
cursor.execute("drop table if exists orders_clean")

In [ ]:
cursor.execute("create table orders_clean as select OrderID,CustomerName,Email,Phone,Phone_valid,City,OrderDate,Amount_Clean as Amount,Amount_Flag,Status from orders_raw")
for row in cursor.fetchall():
  print(row)

In [ ]:
cursor.execute("select count(*) from orders_clean")
for row in cursor.fetchall():
  print(row)

(10000,)


In [ ]:
cursor.execute("select count(*) from(select OrderID from orders_clean group by orderID having count(*))")
for row in cursor.fetchall():
  print(row)


(9700,)


In [ ]:
cursor.execute("select Amount_Flag,count(*) from orders_clean group by Amount_Flag")
cursor.execute("select status,count(*) from orders_clean group by Status order by count(*) Desc")
cursor.execute("select Distinct City from orders_clean order by City")
for row in cursor.fetchall():
  print(row)

(None,)
('  BENGALURU',)
('  BENGALURU ',)
('  BENGALURU  ',)
('  Bengaluru',)
('  Bengaluru ',)
('  Bengaluru  ',)
('  CHENNAI',)
('  CHENNAI ',)
('  CHENNAI  ',)
('  COIMBATORE',)
('  COIMBATORE ',)
('  COIMBATORE  ',)
('  Chennai',)
('  Chennai ',)
('  Chennai  ',)
('  Coimbatore',)
('  Coimbatore ',)
('  Coimbatore  ',)
('  DELHI',)
('  DELHI ',)
('  DELHI  ',)
('  Delhi',)
('  Delhi ',)
('  Delhi  ',)
('  HYDERABAD',)
('  HYDERABAD ',)
('  HYDERABAD  ',)
('  Hyderabad',)
('  Hyderabad ',)
('  Hyderabad  ',)
('  KOLKATA',)
('  KOLKATA ',)
('  KOLKATA  ',)
('  Kolkata',)
('  Kolkata ',)
('  Kolkata  ',)
('  MUMBAI',)
('  MUMBAI ',)
('  MUMBAI  ',)
('  Mumbai',)
('  Mumbai ',)
('  Mumbai  ',)
('  PUNE',)
('  PUNE ',)
('  PUNE  ',)
('  Pune',)
('  Pune ',)
('  Pune  ',)
('  bengaluru',)
('  bengaluru ',)
('  bengaluru  ',)
('  chennai',)
('  chennai ',)
('  chennai  ',)
('  coimbatore',)
('  coimbatore ',)
('  coimbatore  ',)
('  delhi',)
('  delhi ',)
('  delhi  ',)
('  hyderabad',)
